In [4]:
#!gsutil cp -r gs://bucket-for-kpf-project/UCI_Credit_Card.csv ./

In [1]:
!pip install --quiet --upgrade "kfp>=2,<3"
!pip install --quiet google.cloud
!pip install --quiet google-cloud-aiplatform
!pip install -U --quiet google-cloud-pipeline-components

In [1]:
import kfp
import pandas as pd
from google.cloud import aiplatform
from google_cloud_pipeline_components.v1.dataset import TabularDatasetCreateOp
from google_cloud_pipeline_components.v1.automl.training_job import \
    AutoMLTabularTrainingJobRunOp
from google_cloud_pipeline_components.v1.endpoint.create_endpoint.component import \
    endpoint_create as EndpointCreateOp
from google_cloud_pipeline_components.v1.endpoint.deploy_model.component import \
    model_deploy as ModelDeployOp

In [2]:
PROJECT_ID = "project-c8d59959-80e8-496f-96a"
BUCKET_URI = "gs://bucket-for-kpf-project"
DATASET_URI = "gs://bucket-for-kpf-project/UCI_Credit_Card.csv"
GCP_REGION = "us-central1"
DISPLAY_NAME = "credit-card-training-ppl-30Jun26-1"
MACHINE_TYPE = "n1-standard-4"

In [6]:
# Define the workflow of the pipeline.

@kfp.dsl.pipeline(
    name=DISPLAY_NAME,
    pipeline_root=BUCKET_URI)
def pipeline(PROJECT_ID: str):
    ds_op = TabularDatasetCreateOp(
        project=PROJECT_ID,
        display_name=DISPLAY_NAME,
        gcs_source=DATASET_URI
    )
    
    training_op = AutoMLTabularTrainingJobRunOp(
        project=PROJECT_ID,
        location=GCP_REGION,
        display_name=DISPLAY_NAME,
        optimization_prediction_type="classification",
        optimization_objective="minimize-log-loss",
        #budget_milli_node_hours=1000,
        model_display_name=DISPLAY_NAME,
        dataset=ds_op.outputs["dataset"],
        target_column="default.payment.next.month",
    )
    
    endpoint_op = EndpointCreateOp(
        project=PROJECT_ID,
        location=GCP_REGION,
        display_name=DISPLAY_NAME,
    )

    ModelDeployOp(
        model=training_op.outputs["model"],
        endpoint=endpoint_op.outputs["endpoint"],
        dedicated_resources_min_replica_count=1,
        dedicated_resources_max_replica_count=1,
        dedicated_resources_machine_type=MACHINE_TYPE,
    )

In [4]:
kfp.compiler.Compiler().compile(
    pipeline_func=pipeline,
    package_path='card_classif_pipeline.yaml'
)
!gsutil cp -r card_classif_pipeline.yaml gs://bucket-for-kpf-project/

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying file://card_classif_pipeline.yaml [Content-Type=application/yaml]...
/ [1 files][ 42.9 KiB/ 42.9 KiB]                                                
Operation completed over 1 objects/42.9 KiB.                                     


In [5]:

aiplatform.init(
    project=PROJECT_ID,
    location=GCP_REGION,
)

# Prepare the pipeline job
job = aiplatform.PipelineJob(
    display_name="credit-card-pipeline-run",
    template_path="card_classif_pipeline.yaml",
    pipeline_root=BUCKET_URI,
    parameter_values={
        "PROJECT_ID": PROJECT_ID,
    },
    enable_caching=True,
)

job.submit()


Creating PipelineJob
PipelineJob created. Resource name: projects/90514214177/locations/us-central1/pipelineJobs/credit-card-training-ppl-30jun26-1-20260706104244
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/90514214177/locations/us-central1/pipelineJobs/credit-card-training-ppl-30jun26-1-20260706104244')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-central1/pipelines/runs/credit-card-training-ppl-30jun26-1-20260706104244?project=90514214177
